# Compilação do Projeto HydroSense

Este notebook demonstra como compilar o projeto HydroSense usando o sistema de build baseado em CMake e Ninja.
O HydroSense é um sistema IoT para aquicultura baseado em Raspberry Pi Pico W.

## Verificação do Ambiente de Build

Primeiro vamos verificar se temos todas as ferramentas necessárias instaladas.

In [ ]:
import subprocess
import os
import sys

def verificar_ferramenta(comando, nome):
    """Verifica se uma ferramenta está instalada"""
    try:
        resultado = subprocess.run(comando, capture_output=True, text=True, shell=True)
        if resultado.returncode == 0:
            print(f"✅ {nome}: INSTALADO")
            return True
        else:
            print(f"❌ {nome}: NÃO ENCONTRADO")
            return False
    except Exception as e:
        print(f"❌ {nome}: ERRO - {e}")
        return False

print("🔍 Verificando ambiente de build...")
print("="*50)

# Verificar ferramentas essenciais
ferramentas = [
    ("cmake --version", "CMake"),
    ("ninja --version", "Ninja"),
    ("arm-none-eabi-gcc --version", "ARM GCC Compiler"),
    ("python3 --version", "Python 3")
]

todas_ok = True
for comando, nome in ferramentas:
    if not verificar_ferramenta(comando, nome):
        todas_ok = False

print("="*50)
if todas_ok:
    print("🎉 Todas as ferramentas necessárias estão instaladas!")
else:
    print("⚠️  Algumas ferramentas estão faltando. Instale-as antes de continuar.")

## Estrutura do Projeto HydroSense

Vamos examinar a estrutura do projeto para entender os componentes que serão compilados.

In [ ]:
import os
from pathlib import Path

def listar_arvore_diretorio(caminho, nivel=0, max_nivel=2):
    """Lista a estrutura de diretórios do projeto"""
    if nivel > max_nivel:
        return
    
    indent = "  " * nivel
    try:
        items = sorted(os.listdir(caminho))
        for item in items:
            item_path = os.path.join(caminho, item)
            if os.path.isdir(item_path) and not item.startswith('.'):
                print(f"{indent}📁 {item}/")
                listar_arvore_diretorio(item_path, nivel + 1, max_nivel)
            elif item.endswith(('.c', '.h', '.cmake', '.txt', '.md')):
                print(f"{indent}📄 {item}")
    except PermissionError:
        print(f"{indent}❌ Sem permissão para acessar")

# Caminho do projeto
projeto_dir = "/home/hobson007breno/Downloads/projeto final/HydroSense"

print("🏗️  Estrutura do Projeto HydroSense:")
print("="*50)

if os.path.exists(projeto_dir):
    listar_arvore_diretorio(projeto_dir)
else:
    print(f"❌ Diretório do projeto não encontrado: {projeto_dir}")

# Verificar arquivos importantes
arquivos_importantes = [
    "CMakeLists.txt",
    "pico_sdk_import.cmake",
    "src/hydrosense_main.c",
    "src/hydrosense_oled.c",
    "include/hydrosense_system.h"
]

print("\n🔍 Verificando arquivos críticos:")
print("="*50)

for arquivo in arquivos_importantes:
    caminho_completo = os.path.join(projeto_dir, arquivo)
    if os.path.exists(caminho_completo):
        tamanho = os.path.getsize(caminho_completo)
        print(f"✅ {arquivo} ({tamanho} bytes)")
    else:
        print(f"❌ {arquivo} - NÃO ENCONTRADO")

## Configuração do Build

Vamos configurar o sistema de build usando CMake e verificar as dependências.

In [ ]:
import subprocess
import os

def executar_comando(comando, diretorio=None):
    """Executa um comando e retorna o resultado"""
    try:
        print(f"🔨 Executando: {comando}")
        resultado = subprocess.run(
            comando,
            shell=True,
            cwd=diretorio,
            capture_output=True,
            text=True,
            timeout=60
        )
        
        if resultado.returncode == 0:
            print(f"✅ Comando executado com sucesso")
            if resultado.stdout:
                print(f"📄 Saída:\n{resultado.stdout[:500]}..." if len(resultado.stdout) > 500 else f"📄 Saída:\n{resultado.stdout}")
        else:
            print(f"❌ Comando falhou (código: {resultado.returncode})")
            if resultado.stderr:
                print(f"🚨 Erro:\n{resultado.stderr[:500]}..." if len(resultado.stderr) > 500 else f"🚨 Erro:\n{resultado.stderr}")
        
        return resultado
    except subprocess.TimeoutExpired:
        print("⏰ Comando expirou (timeout)")
        return None
    except Exception as e:
        print(f"💥 Erro ao executar comando: {e}")
        return None

# Configurar diretório do projeto
projeto_dir = "/home/hobson007breno/Downloads/projeto final/HydroSense"

print("⚙️  Configurando build com CMake...")
print("="*50)

# Limpar build anterior se existir
build_dir = os.path.join(projeto_dir, "build")
if os.path.exists(build_dir):
    print("🧹 Limpando build anterior...")
    executar_comando("rm -rf build", projeto_dir)

# Criar diretório de build
print("📁 Criando diretório de build...")
executar_comando("mkdir -p build", projeto_dir)

# Configurar com CMake
print("\n⚙️  Executando configuração CMake...")
cmake_resultado = executar_comando(
    "cmake -DCMAKE_BUILD_TYPE=Release -G Ninja ..",
    os.path.join(projeto_dir, "build")
)

if cmake_resultado and cmake_resultado.returncode == 0:
    print("🎉 Configuração CMake concluída com sucesso!")
else:
    print("💔 Falha na configuração CMake")

## Compilação do Projeto

Agora vamos compilar o projeto HydroSense usando Ninja.

In [ ]:
import time
import os

print("🏗️  Iniciando compilação do HydroSense...")
print("="*50)

# Registrar tempo de início
inicio = time.time()

# Executar compilação com Ninja
build_dir = os.path.join(projeto_dir, "build")
ninja_resultado = executar_comando("ninja -v", build_dir)

# Calcular tempo de compilação
fim = time.time()
tempo_compilacao = fim - inicio

print(f"\n⏱️  Tempo de compilação: {tempo_compilacao:.2f} segundos")
print("="*50)

if ninja_resultado and ninja_resultado.returncode == 0:
    print("🎊 COMPILAÇÃO CONCLUÍDA COM SUCESSO!")
    
    # Listar arquivos gerados
    print("\n📦 Arquivos gerados:")
    arquivos_output = [
        "HydroSense.elf",
        "HydroSense.uf2",
        "HydroSense.bin",
        "HydroSense.hex"
    ]
    
    for arquivo in arquivos_output:
        caminho = os.path.join(build_dir, arquivo)
        if os.path.exists(caminho):
            tamanho = os.path.getsize(caminho)
            print(f"  ✅ {arquivo} ({tamanho:,} bytes)")
        else:
            print(f"  ❌ {arquivo} - não encontrado")
    
    print("\n🚀 O firmware está pronto para ser carregado no Raspberry Pi Pico W!")
    print("📋 Use o arquivo HydroSense.uf2 para fazer o flash via USB.")
    
else:
    print("💔 FALHA NA COMPILAÇÃO!")
    print("\n🔧 Possíveis soluções:")
    print("  1. Verificar se todas as dependências estão instaladas")
    print("  2. Verificar se o Pico SDK está configurado corretamente")
    print("  3. Verificar erros de sintaxe no código fonte")
    print("  4. Limpar o build e tentar novamente")

In [ ]:
# Análise detalhada dos componentes compilados
print("📊 Análise dos Componentes Compilados:")
print("="*50)

componentes = {
    "Sistema Principal": ["hydrosense_main.c"],
    "Display OLED": ["hydrosense_oled.c"],
    "Controle de Servo": ["hydrosense_servo.c"],
    "Interface Botões": ["hydrosense_botoes.c"],
    "Utilitários": ["hydrosense_utils.c"],
    "Tasks FreeRTOS": ["tasks/monitoring_task.c", "tasks/feeding_task.c", "tasks/automation_task.c"]
}

for componente, arquivos in componentes.items():
    print(f"\n🔧 {componente}:")
    for arquivo in arquivos:
        caminho = os.path.join(projeto_dir, "src", arquivo)
        if os.path.exists(caminho):
            tamanho = os.path.getsize(caminho)
            linhas = 0
            try:
                with open(caminho, 'r', encoding='utf-8', errors='ignore') as f:
                    linhas = sum(1 for _ in f)
            except:
                linhas = 0
            print(f"  ✅ {arquivo} - {tamanho:,} bytes, ~{linhas} linhas")
        else:
            print(f"  ❌ {arquivo} - não encontrado")

# Verificar tamanho total do firmware
uf2_path = os.path.join(build_dir, "HydroSense.uf2")
if os.path.exists(uf2_path):
    tamanho_fw = os.path.getsize(uf2_path)
    print(f"\n📏 Tamanho total do firmware: {tamanho_fw:,} bytes ({tamanho_fw/1024:.1f} KB)")
    
    # Comparar com limites do Pico
    flash_limit = 2 * 1024 * 1024  # 2MB
    ram_limit = 264 * 1024  # 264KB
    
    percentual_flash = (tamanho_fw / flash_limit) * 100
    
    print(f"💾 Uso da Flash: {percentual_flash:.1f}% do limite do Pico")
    
    if percentual_flash < 50:
        print("✅ Excelente! Muito espaço livre para futuras funcionalidades")
    elif percentual_flash < 75:
        print("✅ Bom! Espaço adequado para expansões")
    elif percentual_flash < 90:
        print("⚠️  Cuidado! Aproximando-se do limite")
    else:
        print("🚨 Crítico! Muito próximo do limite de flash")

print("\n🎯 Compilação do HydroSense concluída!")

## Resumo da Compilação

Este notebook demonstrou o processo completo de compilação do projeto HydroSense:

### 🏗️ **Processo de Build:**
1. **Verificação do ambiente** - CMake, Ninja, ARM GCC
2. **Análise da estrutura** - Código fonte, headers, configuração
3. **Configuração CMake** - Setup do sistema de build
4. **Compilação Ninja** - Build otimizado e rápido
5. **Validação dos arquivos** - Verificação dos outputs gerados

### 📦 **Arquivos Gerados:**
- **HydroSense.uf2** - Arquivo principal para flash USB
- **HydroSense.elf** - Executável com símbolos de debug
- **HydroSense.bin** - Binário puro
- **HydroSense.hex** - Formato Intel HEX

### ⚡ **Funcionalidades Compiladas:**
- Sistema multitarefa FreeRTOS
- Interface OLED SSD1306 melhorada
- Controle de servomotor para alimentação
- Sensores de qualidade da água
- Sistema de automação TPA
- Conectividade WiFi/MQTT

### 🚀 **Próximos Passos:**
1. Conectar o Raspberry Pi Pico W em modo BOOTSEL
2. Copiar HydroSense.uf2 para a unidade USB
3. O sistema será reiniciado automaticamente
4. Monitorar via Serial (115200 baud) para logs